# MaaCare AI: System Evaluation, Design Assumptions & Trade-offs

---

##  1. Core Clinical & Architectural Assumptions

> **The "Sustained Baseline" Assumption:** By aggregating patient metrics over gestational time frames, the model assumes that maternal baseline indicators (like blood pressure trends) act as stable, continuous parameters rather than isolated, fluctuating anomalies.

*   **The Linear Weighting Assumption:** The composite triage matrix prioritizes metric-driven threats over general behavioral feedback. We assume that a patient record exhibiting high physiological anomalies (e.g., BP $140/90$) requires immediate operational escalation over a patient expressing mild physical discomfort (e.g., standard fatigue).
*   **The Representation Assumption:** We assume that metrics like `systolic_bp`, `diastolic_bp`, and `blood_sugar_mgdl` serve as valid digital proxies for accurate, physical maternal health diagnostics in an offline ecosystem.

---

##  2. Discussion of Analytical & Operational Trade-offs

<div style="background-color: #1A1519; padding: 15px; border-left: 5px solid #FF6B8B; border-radius: 4px; margin-bottom: 15px;">
    <strong style="color: #FF6B8B; font-size: 1.1em;"> Immediate Triage vs. Algorithmic Processing</strong><br><br>
    <ul>
        <li><strong>Trade-off:</strong> The controller intercepts specific high-risk text tokens (e.g., *bleeding*, *severe pain*) instantly via regex arrays before passing inputs to the machine learning models.</li>
        <li><strong>The Consequence:</strong> While this guarantees instantaneous safety warnings for severe cases, it bypasses advanced conditional analysis, sacrificing personalized statistical processing in favor of absolute client safety.</li>
    </ul>
</div>

<div style="background-color: #1A1519; padding: 15px; border-left: 5px solid #00DFD8; border-radius: 4px;">
    <strong style="color: #00DFD8; font-size: 1.1em;"> Offline SQLite Storage vs. Cloud Scalability</strong><br><br>
    <ul>
        <li><strong>Trade-off:</strong> Encapsulating the operational dataset completely within an on-device, localized `maacare_offline.db` database.</li>
        <li><strong>The Consequence:</strong> We traded multi-user cloud synchronization and massive global data pooling for localized data privacy, zero-connectivity stability, and instant page rendering speeds.</li>
    </ul>
</div>

---

##  3. Acknowledgment of Structural Limitations

| Diagnostic Axis | Vulnerability Vector | Healthcare System Impact |
| :--- | :--- | :--- |
| **Demographic Invariance** | Uniform threshold metrics | Evaluates a blanket baseline. It does not adjust risk metrics for maternal age, multi-fetal pregnancies, or chronic pre-existing cardiovascular variations. |
| **The "Cold Start" Constraint** | Historical trend dependency | The integrated `MaternalCareEngine` requires at least a baseline set of localized data rows before it can chart meaningful vital trajectories or identify individual variances. |
| **Input Scaling Assumptions** | Text ingestion dependency | Regular expressions and keyword parsing rely on the user clearly explicitly stating symptoms. Vague descriptions might result in standard educational routing instead of necessary escalation. |

---

##  4. Synthesis of Structured System Thinking

To solve this low-connectivity maternal health challenge effectively, the architecture handles metrics through a multi-stage workflow:

In [2]:
import os
import re
import sqlite3
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import gradio as gr
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# =====================================================================
# 0. SYSTEM-WIDE DASHBOARD VISUALIZATION THEME MATRIX (MATERNAL GLOW)
# =====================================================================
sns.set_theme(style="darkgrid") 
plt.rcParams.update({     
    'font.size': 10,     
    'axes.facecolor': '#201A1E',     # Warm deep plum/dark chocolate
    'figure.facecolor': '#140E12',    # Midnight aubergine
    'axes.labelsize': 11,     
    'axes.titlesize': 13,     
    'text.color': '#FFFFFF',     
    'axes.labelcolor': '#FFFFFF',     
    'xtick.color': '#FFFFFF',     
    'ytick.color': '#FFFFFF'
}) 

# =====================================================================
# 1. DATA ACCESS OBJECT (DAO) LAYER [TECHNICAL ARCHITECTURE] 
# =====================================================================
class DataStorageAccessObject: 
    """ 
    Handles offline-first SQLite synchronization, local data normalization,
    and schema verification for patient health logs and guidelines.
    """ 
    @staticmethod
    def initialize_local_database(db_path: str = "maacare_offline.db"):
        """Initializes tables for offline storage of patient metrics and guidelines."""
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Patient Health Metric Log
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS health_logs (
                log_id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
                gestational_week INTEGER,
                systolic_bp INTEGER,
                diastolic_bp INTEGER,
                blood_sugar_mgdl REAL,
                weight_kg REAL,
                reported_symptoms TEXT
            )
        """)
        
        # Clinical Guidance Reference Table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS clinical_guidelines (
                week_number INTEGER PRIMARY KEY,
                fetal_size_equivalent TEXT,
                fetal_development TEXT,
                maternal_changes TEXT,
                nutritional_focus TEXT,
                exercises_allowed TEXT,
                warning_signs TEXT
            )
        """)
        
        # Seed dummy guidance if empty (representing the offline packaged database)
        cursor.execute("SELECT COUNT(*) FROM clinical_guidelines")
        if cursor.fetchone()[0] == 0:
            dummy_guidelines = [
                (12, "Lime", "Baby's reflexes are forming; fingers open and close.", 
                 "Uterus starts expanding out of pelvis. Morning sickness might lessen.", 
                 "Folate, Calcium, Iron-rich dark leafy greens", "Light walking, pelvic floor tilts",
                 "Severe cramping, any vaginal bleeding"),
                (24, "Corn", "Taste buds are developing; lungs starting to form air sacs.",
                 "Visible baby bump. Possible stretch marks and dry skin.",
                 "Hydration, high-fiber foods to combat constipation, Lean Protein", "Prenatal yoga, swimming",
                 "Sudden swelling in face/hands, severe headaches (Preeclampsia risk)"),
                (30, "Cabbage", "Baby is gaining fat; brain is developing rapidly.",
                 "Shortness of breath, backaches, and frequent urination.",
                 "Omega-3 fatty acids, iron, vitamin C for absorption", "Gentle stretching, brief walks",
                 "Decreased fetal movement, rupture of membranes (fluid leaking)"),
                (36, "Romaine Lettuce", "Baby is fully formed and taking up almost all space.",
                 "Increased pelvic pressure as baby descends (lightening).",
                 "Small, frequent nutrient-dense meals to avoid acid reflux", "Slight pelvic rocking, breathing exercises",
                 "Contractions every 5 mins, bright red bleeding, heavy fluid leakage")
            ]
            cursor.executemany("""
                INSERT INTO clinical_guidelines VALUES (?, ?, ?, ?, ?, ?, ?)
            """, dummy_guidelines)
            
        conn.commit()
        conn.close()

    @staticmethod 
    def load_and_sanitize_logs(db_path: str = "maacare_offline.db") -> pd.DataFrame: 
        if not os.path.exists(db_path): 
            DataStorageAccessObject.initialize_local_database(db_path)
            
        conn = sqlite3.connect(db_path)
        df = pd.read_sql_query("SELECT * FROM health_logs", conn)
        conn.close()
        
        # Normalize fields
        df['systolic_bp'] = pd.to_numeric(df['systolic_bp'], errors='coerce').fillna(120)
        df['diastolic_bp'] = pd.to_numeric(df['diastolic_bp'], errors='coerce').fillna(80)
        df['blood_sugar_mgdl'] = pd.to_numeric(df['blood_sugar_mgdl'], errors='coerce').fillna(90)
        df['weight_kg'] = pd.to_numeric(df['weight_kg'], errors='coerce').fillna(60.0)
        
        return df

    @staticmethod
    def insert_health_log(week: int, sys: int, dia: int, sugar: float, weight: float, symptoms: str, db_path: str = "maacare_offline.db"):
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO health_logs (gestational_week, systolic_bp, diastolic_bp, blood_sugar_mgdl, weight_kg, reported_symptoms)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (week, sys, dia, sugar, weight, symptoms))
        conn.commit()
        conn.close()

# =====================================================================
# 2. MODEL LAYER
# =====================================================================
class MaternalCareEngine: 
    """ 
    Source of truth for local pregnancy metrics. Handles offline clinical rules,
    risk prediction modeling, and metrics visualization pipelines.
    """ 
    def __init__(self, validated_dataframe: pd.DataFrame, db_path: str = "maacare_offline.db"): 
        self.df = validated_dataframe 
        self.db_path = db_path
        self.risk_predictor_model = None 
        self._generate_system_kpis() 
        self._build_maternal_dashboards() 
        self._train_predictive_risk_model() 
         
    def _generate_system_kpis(self): 
        self.total_logged_entries = len(self.df) 
        self.latest_week = int(self.df['gestational_week'].iloc[-1]) if len(self.df) > 0 else 12
        self.mean_systolic = round(self.df['systolic_bp'].mean(), 1) if len(self.df) > 0 else 120.0
        self.mean_diastolic = round(self.df['diastolic_bp'].mean(), 1) if len(self.df) > 0 else 80.0
        self.mean_blood_sugar = round(self.df['blood_sugar_mgdl'].mean(), 1) if len(self.df) > 0 else 90.0
         
    def _build_maternal_dashboards(self):
        print(" [Model Layer] Initializing background metrics visualization pipeline...") 
        if self.df.empty:
            # Generate placeholder data if database is brand new to prevent visual crashes
            self.df = pd.DataFrame({
                'gestational_week': [12, 16, 20, 24, 28, 30],
                'systolic_bp': [115, 118, 120, 122, 125, 130],
                'diastolic_bp': [75, 78, 80, 82, 84, 85],
                'blood_sugar_mgdl': [88, 92, 90, 95, 105, 110],
                'weight_kg': [58.0, 59.5, 61.2, 63.0, 65.1, 66.5],
                'reported_symptoms': ["none", "fatigue", "none", "backache", "swelling", "backache"]
            })

        fig, axes = plt.subplots(1, 2, figsize=(14, 5)) 
         
        # 1. Blood Pressure Trends Across Weeks
        sns.lineplot(data=self.df, x='gestational_week', y='systolic_bp', label='Systolic (mmHg)', marker='o', color='#FF6B8B', ax=axes[0]) 
        sns.lineplot(data=self.df, x='gestational_week', y='diastolic_bp', label='Diastolic (mmHg)', marker='s', color='#00DFD8', ax=axes[0]) 
        axes[0].set_title('Systolic & Diastolic Blood Pressure Trends', color='#FF6B8B') 
        axes[0].set_xlabel('Gestational Week')
        axes[0].set_ylabel('Blood Pressure (mmHg)')
        axes[0].legend()
         
        # 2. Blood Sugar Variations
        sns.barplot(data=self.df, x='gestational_week', y='blood_sugar_mgdl', color='#FF9F43', ax=axes[1]) 
        axes[1].axhline(y=140, color='red', linestyle='--', label='GDM Threshold (140 mg/dL)')
        axes[1].set_title('Blood Sugar Tracking (Glucose levels)', color='#FF9F43') 
        axes[1].set_xlabel('Gestational Week')
        axes[1].set_ylabel('Glucose Levels (mg/dL)')
        axes[1].legend()
         
        plt.tight_layout() 
        plt.savefig('maternal_vitals_trends.png', dpi=150) 
        plt.close() 
         
        # 3. Correlation Heatmap between Pregnancy Variables
        plt.figure(figsize=(7, 4.5)) 
        numeric_cols = [c for c in ['gestational_week', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mgdl', 'weight_kg'] if c in self.df.columns] 
        if len(numeric_cols) > 1:
            sns.heatmap(self.df[numeric_cols].corr(), annot=True, cmap='magma', fmt=".2f", cbar=True) 
            plt.title('Inter-Vital Correlation Heatmap', color='#00DFD8') 
            plt.tight_layout() 
            plt.savefig('maternal_correlation_matrix.png', dpi=150) 
            plt.close() 
 
    def _train_predictive_risk_model(self): 
        """
        Trains an offline local classifier assessing risk classification 
        (Normal vs. High Attention) using gestational week and vitals.
        """
        print(" [Model Layer] Training offline predictive clinical classifier...") 
        
        # Synthetic safe reference dataset representing clinical thresholds
        # Features: week, systolic, diastolic, glucose -> Target: risk (0=normal, 1=high risk)
        training_data = pd.DataFrame([
            (12, 110, 70, 85, 0), (24, 115, 75, 90, 0), (32, 120, 80, 95, 0),
            (28, 145, 95, 100, 1), # High Blood Pressure (Preeclampsia risk indicator)
            (26, 118, 76, 145, 1), # High Blood Sugar (Gestational Diabetes indicator)
            (36, 150, 100, 150, 1), # Both elevated
            (20, 130, 85, 110, 0),
            (14, 95, 60, 80, 0),
            (38, 140, 90, 141, 1),
        ], columns=['gestational_week', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mgdl', 'risk_class'])
        
        X = training_data[['gestational_week', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mgdl']]
        y = training_data['risk_class']
         
        preprocessor = ColumnTransformer(transformers=[ 
            ('num', SimpleImputer(strategy='median'), ['gestational_week', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mgdl'])
        ]) 
         
        self.risk_predictor_model = Pipeline(steps=[ 
            ('preprocessor', preprocessor), 
            ('classifier', RandomForestClassifier(n_estimators=15, random_state=42, max_depth=4)) 
        ]) 
        self.risk_predictor_model.fit(X, y) 
        print(" [Model Layer] Offline predictive classifier engine deployment complete.") 
 
    def execute_local_clinical_query(self, query_week: int) -> dict:
        """Retrieves clinical parameters from SQLite database for specified gestational week."""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        # Find closest cataloged week
        cursor.execute("""
            SELECT * FROM clinical_guidelines 
            ORDER BY ABS(week_number - ?) ASC LIMIT 1
        """, (query_week,))
        row = cursor.fetchone()
        conn.close()
        
        if row:
            return {
                "week_matched": row[0],
                "fetal_size": row[1],
                "fetal_development": row[2],
                "maternal_changes": row[3],
                "nutritional_focus": row[4],
                "exercises_allowed": row[5],
                "warning_signs": row[6]
            }
        return {}

    def predict_health_risk(self, week: int, sys: int, dia: int, sugar: float) -> str:
        """Runs the offline ML model to generate safety advice."""
        if self.risk_predictor_model is None:
            return "Indeterminate - Predictive pipeline offline."
            
        test_df = pd.DataFrame([[week, sys, dia, sugar]], columns=['gestational_week', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mgdl'])
        pred = self.risk_predictor_model.predict(test_df)[0]
        prob = self.risk_predictor_model.predict_proba(test_df)[0][1]
        
        if pred == 1 or sys >= 140 or dia >= 90 or sugar >= 140:
            return f"HIGH RISK ALERT (Confidence: {prob*100:.1f}%)\n" \
                   f"Please coordinate with your OB/GYN immediately.\n" \
                   f"Guidance: Reduce sodium intake, track fetal movement, monitor for severe headaches."
        else:
            return f"🟢 VITALS NORMAL (Confidence: {(1-prob)*100:.1f}%)\n" \
                   f"Your current logs align nicely with expected cardiovascular and metabolic trends."

# =====================================================================
# 3. CONTROLLER LAYER 
# =====================================================================
class StatefulMaternalIntentController: 
    """ 
    Decodes pregnancy query contexts, scans clinical warnings, 
    and returns localized responses paired with actionable guidance.
    """ 
    def __init__(self, data_engine: MaternalCareEngine):
        self.engine = data_engine

    def parse_and_route_query(self, user_text: str) -> str:
        tokens = user_text.lower().strip()
        print(f"[Controller LOG] Processing Query: '{user_text}'")
        
        # 1. EMERGENCY TRIAGE LAYER (Highest Priority)
        emergency_triggers = ["bleeding", "severe pain", "cramp", "seizure", "no movement", "leak", "headache", "blurry", "preeclampsia"]
        if any(trigger in tokens for trigger in emergency_triggers):
            return (
                " **EMERGENCY MEDICAL WARNING - IMMEDIATE INTERVENTION REQUIRED** \n\n"
                "The symptoms you typed may indicate an acute obstetric complication (such as preeclampsia or early onset labor).\n\n"
                "**Please act immediately:**\n"
                "1. DO NOT wait for further offline responses.\n"
                "2. Travel to the nearest Emergency Obstetric Care Facility or contact your primary healthcare provider.\n"
                "3. If bleeding or suffering severe abdominal pain, lie down on your left side to maximize uterine blood flow."
            )

        # 2. GREETINGS LAYER
        if any(greet in tokens for greet in ["hi", "hello", "hey", "greetings", "good morning", "good afternoon"]):
            return (
                f" **Hello! Welcome to MaaCare AI - Your Privacy-First Offline Pregnancy Assistant.** 🌸\n\n"
                f"I am initialized with clinical data records tracking pregnancy vitals and milestones.\n"
                f"You are currently at **Week {self.engine.latest_week}** based on your latest data.\n\n"
                f"How can I support you or what metrics would you like to update?"
            )

        # 3. GESTATIONAL WEEK MATCHING DETECTOR
        week_match = re.search(r"(\d+)\s*(?:weeks|wk|week)", tokens)
        if week_match:
            query_week = int(week_match.group(1))
            guidance = self.engine.execute_local_clinical_query(query_week)
            if guidance:
                return (
                    f"📅 **MaaCare Week-by-Week Milestone Dashboard [Week {query_week}]**\n"
                    f"========================================================\n"
                    f"🍒 **Fetal Size Equivalent:** Like a **{guidance['fetal_size']}**\n\n"
                    f"👶 **Fetal Development:** {guidance['fetal_development']}\n\n"
                    f"🤰 **Maternal Bodily Changes:** {guidance['maternal_changes']}\n\n"
                    f"🥦 **Nutritional Recommendation:** Focus on *{guidance['nutritional_focus']}*\n\n"
                    f"🏃‍♀️ **Permissible Safe Exercises:** {guidance['exercises_allowed']}\n\n"
                    f"⚠️ **Targeted Warning Signs:** {guidance['warning_signs']}\n"
                    f"--------------------------------------------------------\n"
                    f"*Note: Close matching guidance for Week {guidance['week_matched']} was pulled from local storage.*"
                )

        # 4. DIET & NUTRITION DETECTOR
        if any(nutri in tokens for nutri in ["eat", "food", "diet", "nutrition", "avoid", "iron", "calcium"]):
            return (
                "🥦 **MaaCare Nutrition & Safe Foods Matrix**\n\n"
                "During pregnancy, nutritional density supports both fetal organogenesis and maternal volume expansion.\n\n"
                "*   **Safe Essentials:**\n"
                "    - **Iron:** Lean meats, spinach, lentils, paired with Vitamin C (citrus) to maximize bioavailability.\n"
                "    - **Calcium:** Yogurt, milk, fortified plant milks, and paneer for fetal bone structure mineralization.\n"
                "    - **Folate:** Dark leafy greens, oranges, whole grains to support neural tube development.\n"
                "*   ❌ **STRICTLY AVOID:** Unpasteurized dairy, raw eggs (risk of Listeria/Salmonella), high-mercury fish (swordfish/king mackerel), and raw sprouts."
            )

        # 5. VITALS ASSESSMENT ENGINE
        if any(vital in tokens for vital in ["blood pressure", "sugar", "diabetes", "hypertension", "bp", "vitals"]):
            # Extract values if present or default to latest database stats
            sys_val = self.engine.mean_systolic
            dia_val = self.engine.mean_diastolic
            sugar_val = self.engine.mean_blood_sugar
            
            # Regex patterns for custom vitals inside query
            bp_match = re.search(r"(\d+)\s*/\s*(\d+)", tokens)
            sugar_match = re.search(r"sugar\s*(?:of|is)?\s*(\d+)", tokens)
            
            if bp_match:
                sys_val = int(bp_match.group(1))
                dia_val = int(bp_match.group(2))
            if sugar_match:
                sugar_val = float(sugar_match.group(1))
                
            prediction = self.engine.predict_health_risk(self.engine.latest_week, sys_val, dia_val, sugar_val)
            return (
                f" **MaaCare Real-time Intelligent Health Assessment**\n"
                f"--------------------------------------------------\n"
                f"Evaluated Metrics: BP: **{sys_val}/{dia_val} mmHg** | Glucose: **{sugar_val} mg/dL**\n\n"
                f"**Clinical Status:**\n{prediction}\n\n"
                f"*Medical Disclaimer: The on-device classifier is an educational metric and does not substitute for regular clinical urine protein analyses or diagnostic GTTs.*"
            )

        # 6. FAIRY TALE/UNKNOWN FALLBACK
        return (
            "**MaaCare AI Care Guide**\n\n"
            "I heard your inquiry but couldn't map it directly. You can ask me:\n"
            "1. 'What should I eat?' (Dietary support)\n"
            "2. 'Is my BP 130/85 normal?' (Vitals classification)\n"
            "3. 'What happens in week 24 of pregnancy?' (Milestone updates)\n"
            "4. 'How can I treat severe abdominal cramps?' (Emergency protocols)"
        )

# =====================================================================
# 4. VIEW (GRADIO INTERFACE UI) LAYER
# =====================================================================
class MaaCareGlowDashboardView: 
    def __init__(self, architecture_controller: StatefulMaternalIntentController): 
        self.controller = architecture_controller 
        self._assemble_ui_workspace() 
         
    def _assemble_ui_workspace(self): 
        with gr.Blocks() as self.workspace_root: 
            gr.Markdown("<h1 style='text-align: center; color: #FF6B8B;'> MaaCare AI: Offline Pregnancy Health Assistant</h1>") 
            gr.Markdown("<p style='text-align: center; color: #E0D1DA;'>Privacy-First Maternal Care & Automated Local Diagnostics (Offline-Ready)</p>")
             
            with gr.Row(): 
                # Column 1: Client logs and interaction terminal
                with gr.Column(scale=3): 
                    gr.Markdown("### 🩺 Update Maternal Daily Health Logs") 
                    
                    with gr.Row():
                        gest_week_input = gr.Slider(minimum=1, maximum=42, value=12, step=1, label="Gestational Week")
                        weight_input = gr.Number(value=60.0, label="Maternal Weight (kg)")
                    
                    with gr.Row():
                        systolic_bp_input = gr.Number(value=120, label="Systolic BP (mmHg)")
                        diastolic_bp_input = gr.Number(value=80, label="Diastolic BP (mmHg)")
                        sugar_input = gr.Number(value=95.0, label="Fasting Blood Sugar (mg/dL)")
                    
                    reported_symptoms_input = gr.Textbox(placeholder="e.g., fatigue, back pain, none", label="Reported Physical Symptoms")
                    commit_log_btn = gr.Button("Save Daily Health Logs Offline", variant="primary")
                    
                    gr.Markdown("---")
                    gr.Markdown("###  Clinical AI Care Agent") 
                    live_chat_terminal = gr.Chatbot(label="MaaCare Advisor Feed", height=320) 
                    input_text_field = gr.Textbox(placeholder="Ask anything: e.g. 'What can I eat during my second trimester?' or 'Is BP 145/95 safe?'", label="Query Input") 
                    fire_execution_btn = gr.Button("Submit Offline Query") 
                     
                # Column 2: Dashboard Visualization Pipeline
                with gr.Column(scale=3): 
                    gr.Markdown("###  Patient Offline Clinical Analytics") 
                    vitals_plot = gr.Image("maternal_vitals_trends.png", label="Vitals Trajectory Across Weeks") 
                    correlation_plot = gr.Image("maternal_correlation_matrix.png", label="Inter-Vital Correlation Matrix") 
                    
            # Set action listeners
            commit_log_btn.click(
                fn=self._save_log_and_update_metrics, 
                inputs=[gest_week_input, systolic_bp_input, diastolic_bp_input, sugar_input, weight_input, reported_symptoms_input],
                outputs=[vitals_plot, correlation_plot, live_chat_terminal]
            )
            
            fire_execution_btn.click(
                fn=self._transaction_routing_pipeline, 
                inputs=[input_text_field, live_chat_terminal], 
                outputs=[input_text_field, live_chat_terminal]
            ) 
            input_text_field.submit(
                fn=self._transaction_routing_pipeline, 
                inputs=[input_text_field, live_chat_terminal], 
                outputs=[input_text_field, live_chat_terminal]
            ) 
             
    def _save_log_and_update_metrics(self, week, sys, dia, sugar, weight, symptoms):
        # 1. Write data points into database
        DataStorageAccessObject.insert_health_log(int(week), int(sys), int(dia), float(sugar), float(weight), str(symptoms))
        
        # 2. Reload sanitized dataframe
        updated_df = DataStorageAccessObject.load_and_sanitize_logs()
        
        # 3. Reload calculations & visualizations in the core engine
        self.controller.engine.df = updated_df
        self.controller.engine._generate_system_kpis()
        self.controller.engine._build_maternal_dashboards()
        self.controller.engine._train_predictive_risk_model()
        
        # 4. Generate system notice
        save_notification = [
            {"role": "user", "content": "Database Commit Triggered"},
            {"role": "assistant", "content": f" **System Success:** Local SQLite Database Updated! New health record registered at **Week {week}**. Visual analytics recalculated."}
        ]
        
        return "maternal_vitals_trends.png", "maternal_correlation_matrix.png", save_notification

    def _transaction_routing_pipeline(self, user_msg, chat_history): 
        if not user_msg.strip(): 
            return "", chat_history 
        if chat_history is None:
            chat_history = []
        try:     
            agent_reply = self.controller.parse_and_route_query(user_msg)
            chat_history.append({"role": "user", "content": user_msg})
            chat_history.append({"role": "assistant", "content": agent_reply})
        except Exception as e:     
            chat_history.append({"role": "assistant", "content": f"Critical Error: {str(e)}"})
        return "", chat_history 
       
    def run_production_server(self): 
        print(" [System Deployment] Deploying MaaCare Local Port Interface...") 
        self.workspace_root.launch(
            theme=gr.themes.Monochrome(),
            share=True, 
            prevent_thread_lock=True
        ) 

# =====================================================================
# 5. LIFE-CYCLE RUN INITIALIZATION NODE 
# =====================================================================
if __name__ == "__main__": 
    # Create local storage files if not existing
    DataStorageAccessObject.initialize_local_database()
    
    # Run pipeline loading
    sanitized_df = DataStorageAccessObject.load_and_sanitize_logs() 
    engine = MaternalCareEngine(sanitized_df) 
    controller = StatefulMaternalIntentController(engine) 
    view = MaaCareGlowDashboardView(controller) 

    # Activate Gradio engine
    view.run_production_server()

 [Model Layer] Initializing background metrics visualization pipeline...
 [Model Layer] Training offline predictive clinical classifier...
 [Model Layer] Offline predictive classifier engine deployment complete.
 [System Deployment] Deploying MaaCare Local Port Interface...
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://32fefbb90eec824cf4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
